# Store files in R2 and continue the learning experiment

First use a CPU session with `MODE = "COPY"`. This copies and checks the experiment files from Drive. It does no training.

Then open a fresh A100 session with `MODE = "RUN"`. This loads your saved model after task 5, teaches only task 17, and saves results to R2 automatically. The learning experiment is unchanged.

Before starting, follow the [one-time R2 setup](https://github.com/sumitasthana/CARK/blob/main/docs/R2_STORAGE.md). A bucket is a named place for files. Add the three values to Colab Secrets, using the key icon in the sidebar. Do not paste credentials into cells.

## 1. Load the code

Run this cell in both sessions. It loads a fixed code version and the file-transfer library.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
REPO = Path("/content/uncle-r2-learning")
CODE_VERSION = "432863803df1e0312cc0f98adad9be04c5bfcd67"
assert not any(n == "uncle" or n.startswith("uncle.") for n in sys.modules), "Start a fresh runtime before loading this code again."
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), "Use a fresh session."
    remote = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip().removesuffix(".git")
    assert remote == "https://github.com/sumitasthana/CARK", "This folder belongs to another project."
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Save local code changes and use a fresh session."
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "boto3>=1.36"], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Code version:", CODE_VERSION)

## 2. Connect to your bucket

Leave COPY for the first CPU session. Change it to RUN only after the copy is checked. If your bucket has another name, change BUCKET. The source file below is the model you already confirmed; there is no file number to choose.

Use a new EXPERIMENT_ID if a training session was interrupted or if you change beta. Start with beta 0.01. You can try more values later, after reading this first result.

In [ ]:
import json
import math
import torch
from google.colab import userdata
from uncle.storage import R2Store
from uncle.learning_diagnostic import inspect_source, file_hash, run_learning_diagnostic, summarize_diagnostic

MODE = "COPY"  # CPU: COPY first. Fresh A100: RUN afterwards.
BUCKET = "uncle-experiments"
EXPERIMENT_ID = "20261006_learning_loss_r2_01"
BETAS = (0.01,)
SOURCE_KEY = "uncle/learning_initialization/20261006_hyperfan_L3_L0_L9_seed0_02/checkpoint_seq1_resnet50_seed0.pt"
DRIVE_ROOT = Path("/content/drive/MyDrive/uncle")
COPY_PREFIX = "uncle"
CACHE = Path("/content/r2-cache")
OUTPUT_ROOT = Path("/content/r2-results") / EXPERIMENT_ID
OUTPUT_PREFIX = "uncle/learning_loss_diagnostic/" + EXPERIMENT_ID
DATA = Path("/content/data/tiny-imagenet-200")
NEW_TASK = "17"
assert MODE in ("COPY", "RUN")
assert EXPERIMENT_ID and all(c.isalnum() or c in "_-" for c in EXPERIMENT_ID)
assert BETAS and len(set(BETAS)) == len(BETAS)
assert all(math.isfinite(b) and b >= 0 for b in BETAS)
store = R2Store.connect(userdata.get("R2_ENDPOINT"), BUCKET,
    userdata.get("R2_ACCESS_KEY_ID"), userdata.get("R2_SECRET_ACCESS_KEY"))
# Listing one page checks that the bucket exists and the credentials work.
next(iter(store.keys(COPY_PREFIX + "/")), None)
print("Connected to R2 bucket:", BUCKET)
print("Mode:", MODE)

## 3. Copy Drive files once, then check the source model

In COPY mode, allow Drive access. All files under DRIVE_ROOT are copied and read back to check their contents. Originals stay on Drive. If the copy stops, run COPY again; matching files are checked instead of uploaded again. Close other notebooks that are saving to this folder first.

In RUN mode, this cell downloads only the chosen model. It checks the tasks and scores saved inside it. It stops if this is a different model.

In [ ]:
if MODE == "COPY":
    assert not torch.cuda.is_available(), "Use a CPU session for the one-time copy."
    from google.colab import drive
    drive.mount("/content/drive")
    def copied(row):
        print("Copied and checked:", row["key"], flush=True)
    copied_files = store.copy_tree(DRIVE_ROOT, COPY_PREFIX, on_file=copied)
    print("Copy finished:", copied_files["files"], "files checked.")
else:
    assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0), "Choose a fresh A100 runtime."
    assert not list(store.keys(OUTPUT_PREFIX + "/")), "This experiment has saved files. Read them first; use a new experiment name for a new run."
SOURCE = store.download(SOURCE_KEY, CACHE)
details = inspect_source(SOURCE)
assert details["completed_tasks"] == ["3", "0", "9", "5"], "Choose the confirmed model after task 5."
assert details["config"]["initialization"] == "hyperfan_in"
assert details["config"]["dataset"] == "tiny_imagenet" and details["config"]["backbone"] == "resnet50"
assert details["accuracies"] == {"3": 27.0, "0": 31.2, "9": 48.4, "5": 54.8}, "Saved scores differ from your confirmed model. Stop and check the source."
SOURCE_SHA256 = file_hash(SOURCE)
print("Saved tasks:", details["completed_tasks"])
print("Saved validation scores:", details["accuracies"])
print("Model fingerprint:", SOURCE_SHA256)
print("Next lesson: task", NEW_TASK, "only.")

## 4. Teach task 17 and save results automatically

COPY mode skips this cell. RUN mode checks the starting scores on validation images, then teaches task 17. After every pass through the training images, it prints losses and scores and uploads the report. At the end it uploads and checks the final model, then saves the completed report.

Every beta loads the same source model independently. If Colab stops during training, the uploaded reports remain available. A final model is saved only when the lesson finishes. This notebook cannot continue midway through an interrupted lesson.

In [ ]:
reports = []
if MODE == "RUN":
    from uncle.config import Config
    from uncle.streams import build_tasks
    assert file_hash(SOURCE) == SOURCE_SHA256
    source_config = Config(**details["config"])
    task_order = (*details["completed_tasks"], NEW_TASK)
    tasks = build_tasks(source_config, root=DATA, include=task_order, download=True)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    selection = {"source_key": SOURCE_KEY, "source_sha256": SOURCE_SHA256,
        "completed_tasks": details["completed_tasks"], "new_task": NEW_TASK,
        "saved_accuracies": details["accuracies"], "code_version": CODE_VERSION, "betas": BETAS}
    (OUTPUT_ROOT / "selection.json").write_text(json.dumps(selection, indent=2) + "\n")
    store.upload(OUTPUT_ROOT / "selection.json", OUTPUT_PREFIX + "/selection.json")
    for beta in BETAS:
        label = format(beta, ".8g").replace(".", "_")
        output = OUTPUT_ROOT / f"beta_{label}"
        prefix = OUTPUT_PREFIX + f"/beta_{label}"
        def save_epoch(row):
            store.publish_lesson(output, prefix)
            print(f"Pass {row['epoch']}: new-task loss = {row['task_loss']:.4f}; beta times protection = {row['weighted_protection_loss']:.4f}")
            print("Validation scores:", " | ".join(f"task {t}: {row['accuracies'][t]:.1f}%" for t in task_order), flush=True)
            print("Report checked in R2.", flush=True)
        print("Loading the source model. Beta:", beta, flush=True)
        report = run_learning_diagnostic(SOURCE, output, NEW_TASK, tasks,
            beta=beta, device="cuda", code_version=CODE_VERSION, on_epoch=save_epoch)
        assert report["source_sha256"] == SOURCE_SHA256
        store.publish_lesson(output, prefix, complete=True)
        reports.append((output, prefix, report))
        print("Final model and report checked in R2:", prefix, flush=True)
        torch.cuda.empty_cache()
    comparisons = [summarize_diagnostic(report) for _, _, report in reports]
    (OUTPUT_ROOT / "beta_comparison.json").write_text(json.dumps(comparisons, indent=2) + "\n")
    store.upload(OUTPUT_ROOT / "beta_comparison.json", OUTPUT_PREFIX + "/beta_comparison.json")
    for row in comparisons:
        print(f"Beta {row['beta']:g}: task 17 = {row['new_task_accuracy']:.1f}%; mean old score change = {row['mean_old_change']:+.1f} points")
    print("Training and uploads finished. Run the final cell to check history and release the GPU.")
else:
    print("COPY mode: no training. Run the final cell to finish the CPU session.")

## 5. Check saved history and close the session

RUN mode checks that the final model adds exactly one lesson and preserves the earlier history. The model and report have already been checked in R2. After these checks, the GPU is released.

COPY mode disconnects Drive. Then open a fresh A100 session, change MODE to RUN, and run from the top. Keep the same bucket and source file.

In [ ]:
from google.colab import runtime
if MODE == "RUN":
    from uncle import checkpoint as checkpointing
    assert reports and len(reports) == len(BETAS)
    for output, prefix, report in reports:
        saved_report = json.loads((output / "report.json").read_text())
        assert saved_report == report and report["status"] == "complete"
        saved_config = dict(report["config"])
        saved_config["tasks"] = tuple(saved_config["tasks"])
        saved_config["hidden"] = tuple(saved_config["hidden"])
        saved_config["requests"] = tuple(tuple(r) for r in saved_config["requests"])
        final = checkpointing.load(output / "checkpoint.pt", Config(**saved_config))
        assert len(final["history"]) == len(report["source_history"]) + 1
        assert json.loads(json.dumps(final["history"][:-1])) == report["source_history"]
        assert final["history"][-1]["task"] == NEW_TASK
        assert final["previous"] == report["final_accuracies"]
        assert len(report["epochs"]) == report["config"]["epochs"]
        assert not final["forgotten"]
        remote_report = store.download(prefix + "/report.json", CACHE)
        assert json.loads(remote_report.read_text()) == report
        remote_model = store.info(prefix + "/checkpoint.pt")
        assert remote_model["Metadata"]["sha256"] == file_hash(output / "checkpoint.pt")
        print("Checked history and R2 files:", prefix)
        del final
    print("Files checked in R2. Ending the GPU session.", flush=True)
else:
    assert copied_files["files"] > 0
    print("Drive files copied and checked. Next: fresh A100 session, MODE = RUN.", flush=True)
    drive.flush_and_unmount(timeout_ms=600000)
if MODE == "RUN":
    runtime.unassign()